# Actividad 3 Individual — Desarrollo de un Problema de Planificación

**Maestría en Inteligencia Artificial Aplicada**  
**Toma de Decisiones**  

---

## Objetivos

Aplicar herramientas de planificación automatizada para resolver un problema descrito en lenguaje **PDDL (Planning Domain Definition Language)**, haciendo uso del planificador ganador de la **IPC2018 (International Planning Competition 2018)**.

---

## 1. Contexto: IPC2018 y Planificador Ganador

La **International Conference on Automated Planning and Scheduling (ICAPS)** organiza competencias donde equipos de investigación compiten con planificadores automáticos en dominios PDDL estandarizados.  
Referencia: https://www.icaps-conference.org/competitions/

En la edición **IPC2018 – Classical Tracks** (https://ipc2018-classical.bitbucket.io/), el planificador ganador del **Optimal Track** fue **Scorpion** (del equipo de Jendrik Seipp et al., basado en Fast Downward con heurística SCP).

El repositorio de contenedores Singularity con los planificadores está disponible en la página de la competencia. Para ejecutar el planificador ganador sobre los archivos PDDL generados en esta actividad se utiliza Singularity en un entorno Linux (o WSL en Windows).

---

## 2. Descripción del Problema

Un **robot rover** realizó previamente la excavación de dos rocas en:
- **Localidad 1** → mineral_1
- **Localidad 2** → mineral_2

El objetivo es generar el **plan** que debe seguir el rover para llevar ambos minerales al **laboratorio de análisis** ubicado en la **Localidad 5**.

### Restricciones del terreno

| Conexión | Tipo |
|---|---|
| Localidad 3 ↔ Localidad 1 | Bidireccional (libre) |
| Localidad 3 → Localidad 2 | Una sola dirección |
| Localidad 2 → Localidad 4 | Una sola dirección |
| Localidad 3 ↔ Localidad 4 | Bidireccional |
| Localidad 4 ↔ Localidad 5 | Bidireccional |

```
        [loc1]
         ↑↓
[loc2] ← [loc3] ↔ [loc4] ↔ [loc5 = LAB]
   ↓              ↑
   └──────────────┘
```

El rover inicia su trayecto en la **Localidad 3**.

---

## 3. Dominio PDDL — `domain.pddl`

El dominio modela:
- **Tipos**: rover, localidad, mineral
- **Predicados**: posición del rover, presencia de mineral en localidad, si el rover transporta un mineral, si el laboratorio está en una localidad, si el mineral fue analizado, y si existe camino entre localidades.
- **Acciones**: mover, recoger mineral, entregar mineral en laboratorio.

### Por qué este diseño

- El rover puede **cargar un solo mineral a la vez** (realista para rovers de exploración).
- Las rutas son asimétricas, lo que requiere predicados `(camino ?l1 ?l2)` unidireccionales.
- La acción `entregar` requiere que el rover esté en una localidad con laboratorio.

In [1]:
domain_pddl = """
(define (domain rover-minerales)
  (:requirements :typing :negative-preconditions)

  (:types
    localidad mineral rover - object
  )

  (:predicates
    (en-rover ?r - rover ?l - localidad)        ; el rover está en localidad l
    (en-mineral ?m - mineral ?l - localidad)    ; el mineral m está en localidad l
    (transportando ?r - rover ?m - mineral)     ; el rover lleva el mineral m
    (laboratorio ?l - localidad)                ; localidad l tiene laboratorio
    (analizado ?m - mineral)                    ; el mineral m ya fue analizado
    (camino ?l1 - localidad ?l2 - localidad)    ; existe camino de l1 a l2
    (manos-libres ?r - rover)                   ; el rover no carga nada
  )

  ; ------------------------------------------------------------------
  ; ACCIÓN: Mover el rover de una localidad a otra
  ; ------------------------------------------------------------------
  (:action mover
    :parameters (?r - rover ?desde - localidad ?hacia - localidad)
    :precondition (and
      (en-rover ?r ?desde)
      (camino ?desde ?hacia)
    )
    :effect (and
      (not (en-rover ?r ?desde))
      (en-rover ?r ?hacia)
    )
  )

  ; ------------------------------------------------------------------
  ; ACCIÓN: Recoger un mineral en la localidad donde está el rover
  ; ------------------------------------------------------------------
  (:action recoger
    :parameters (?r - rover ?m - mineral ?l - localidad)
    :precondition (and
      (en-rover ?r ?l)
      (en-mineral ?m ?l)
      (manos-libres ?r)
    )
    :effect (and
      (transportando ?r ?m)
      (not (en-mineral ?m ?l))
      (not (manos-libres ?r))
    )
  )

  ; ------------------------------------------------------------------
  ; ACCIÓN: Entregar mineral en el laboratorio
  ; ------------------------------------------------------------------
  (:action entregar
    :parameters (?r - rover ?m - mineral ?l - localidad)
    :precondition (and
      (en-rover ?r ?l)
      (transportando ?r ?m)
      (laboratorio ?l)
    )
    :effect (and
      (not (transportando ?r ?m))
      (manos-libres ?r)
      (analizado ?m)
    )
  )
)
"""
print(domain_pddl)


(define (domain rover-minerales)
  (:requirements :typing :negative-preconditions)

  (:types
    localidad mineral rover - object
  )

  (:predicates
    (en-rover ?r - rover ?l - localidad)        ; el rover está en localidad l
    (en-mineral ?m - mineral ?l - localidad)    ; el mineral m está en localidad l
    (transportando ?r - rover ?m - mineral)     ; el rover lleva el mineral m
    (laboratorio ?l - localidad)                ; localidad l tiene laboratorio
    (analizado ?m - mineral)                    ; el mineral m ya fue analizado
    (camino ?l1 - localidad ?l2 - localidad)    ; existe camino de l1 a l2
    (manos-libres ?r - rover)                   ; el rover no carga nada
  )

  ; ------------------------------------------------------------------
  ; ACCIÓN: Mover el rover de una localidad a otra
  ; ------------------------------------------------------------------
  (:action mover
    :parameters (?r - rover ?desde - localidad ?hacia - localidad)
    :precondition 

In [2]:
# Escribir el archivo domain.pddl
with open('domain.pddl', 'w', encoding='utf-8') as f:
    f.write(domain_pddl.strip())
print('domain.pddl generado correctamente.')

domain.pddl generado correctamente.


---

## 4. Problema 1 — Escenario original: dos minerales, cinco localidades

### Estado inicial
- Rover en Localidad 3
- Mineral 1 en Localidad 1
- Mineral 2 en Localidad 2
- Laboratorio en Localidad 5

### Meta
- `mineral_1` analizado
- `mineral_2` analizado

### Plan esperado (traza)

```
1.  mover rover1 loc3 loc1
2.  recoger rover1 mineral1 loc1
3.  mover rover1 loc1 loc3
4.  mover rover1 loc3 loc4
5.  mover rover1 loc4 loc5
6.  entregar rover1 mineral1 loc5
7.  mover rover1 loc5 loc4
8.  mover rover1 loc4 loc3
9.  mover rover1 loc3 loc2
10. recoger rover1 mineral2 loc2
11. mover rover1 loc2 loc4
12. mover rover1 loc4 loc5
13. entregar rover1 mineral2 loc5
```

In [3]:
problem1_pddl = """
(define (problem rover-problema1)
  (:domain rover-minerales)

  (:objects
    rover1 - rover
    loc1 loc2 loc3 loc4 loc5 - localidad
    mineral1 mineral2 - mineral
  )

  (:init
    ; Posición inicial del rover
    (en-rover rover1 loc3)
    (manos-libres rover1)

    ; Minerales excavados
    (en-mineral mineral1 loc1)
    (en-mineral mineral2 loc2)

    ; Laboratorio
    (laboratorio loc5)

    ; Red de caminos
    ;   loc3 <-> loc1  (bidireccional)
    (camino loc3 loc1)
    (camino loc1 loc3)
    ;   loc3 -> loc2   (una dirección)
    (camino loc3 loc2)
    ;   loc2 -> loc4   (una dirección)
    (camino loc2 loc4)
    ;   loc3 <-> loc4  (bidireccional)
    (camino loc3 loc4)
    (camino loc4 loc3)
    ;   loc4 <-> loc5  (bidireccional)
    (camino loc4 loc5)
    (camino loc5 loc4)
  )

  (:goal
    (and
      (analizado mineral1)
      (analizado mineral2)
    )
  )
)
"""
print(problem1_pddl)


(define (problem rover-problema1)
  (:domain rover-minerales)

  (:objects
    rover1 - rover
    loc1 loc2 loc3 loc4 loc5 - localidad
    mineral1 mineral2 - mineral
  )

  (:init
    ; Posición inicial del rover
    (en-rover rover1 loc3)
    (manos-libres rover1)

    ; Minerales excavados
    (en-mineral mineral1 loc1)
    (en-mineral mineral2 loc2)

    ; Laboratorio
    (laboratorio loc5)

    ; Red de caminos
    ;   loc3 <-> loc1  (bidireccional)
    (camino loc3 loc1)
    (camino loc1 loc3)
    ;   loc3 -> loc2   (una dirección)
    (camino loc3 loc2)
    ;   loc2 -> loc4   (una dirección)
    (camino loc2 loc4)
    ;   loc3 <-> loc4  (bidireccional)
    (camino loc3 loc4)
    (camino loc4 loc3)
    ;   loc4 <-> loc5  (bidireccional)
    (camino loc4 loc5)
    (camino loc5 loc4)
  )

  (:goal
    (and
      (analizado mineral1)
      (analizado mineral2)
    )
  )
)



In [4]:
with open('problem1.pddl', 'w', encoding='utf-8') as f:
    f.write(problem1_pddl.strip())
print('problem1.pddl generado correctamente.')

problem1.pddl generado correctamente.


---

## 5. Problema 2 — Escenario propuesto: tercer mineral y laboratorio secundario

### Escenario extendido

Se añaden:
- **Localidad 6**: nueva zona de excavación con `mineral_3`.
- **Localidad 7**: laboratorio secundario (laboratorio de campo).
- Conexiones nuevas: `loc5 → loc6` (una dirección) y `loc6 ↔ loc7` (bidireccional).

### Meta
Los tres minerales deben estar analizados. Se permite entregarlos en cualquiera de los dos laboratorios.

```
[loc1] ↔ [loc3] ↔ [loc4] ↔ [loc5=LAB1] → [loc6]
           ↓          ↑                      ↔
         [loc2] ───────┘                   [loc7=LAB2]
```

In [5]:
problem2_pddl = """
(define (problem rover-problema2)
  (:domain rover-minerales)

  (:objects
    rover1 - rover
    loc1 loc2 loc3 loc4 loc5 loc6 loc7 - localidad
    mineral1 mineral2 mineral3 - mineral
  )

  (:init
    ; Posición inicial del rover
    (en-rover rover1 loc3)
    (manos-libres rover1)

    ; Minerales excavados
    (en-mineral mineral1 loc1)
    (en-mineral mineral2 loc2)
    (en-mineral mineral3 loc6)

    ; Laboratorios
    (laboratorio loc5)
    (laboratorio loc7)

    ; Red de caminos (original)
    (camino loc3 loc1)
    (camino loc1 loc3)
    (camino loc3 loc2)
    (camino loc2 loc4)
    (camino loc3 loc4)
    (camino loc4 loc3)
    (camino loc4 loc5)
    (camino loc5 loc4)

    ; Nuevas conexiones
    (camino loc5 loc6)    ; una sola dirección
    (camino loc6 loc7)    ; bidireccional
    (camino loc7 loc6)
  )

  (:goal
    (and
      (analizado mineral1)
      (analizado mineral2)
      (analizado mineral3)
    )
  )
)
"""
print(problem2_pddl)


(define (problem rover-problema2)
  (:domain rover-minerales)

  (:objects
    rover1 - rover
    loc1 loc2 loc3 loc4 loc5 loc6 loc7 - localidad
    mineral1 mineral2 mineral3 - mineral
  )

  (:init
    ; Posición inicial del rover
    (en-rover rover1 loc3)
    (manos-libres rover1)

    ; Minerales excavados
    (en-mineral mineral1 loc1)
    (en-mineral mineral2 loc2)
    (en-mineral mineral3 loc6)

    ; Laboratorios
    (laboratorio loc5)
    (laboratorio loc7)

    ; Red de caminos (original)
    (camino loc3 loc1)
    (camino loc1 loc3)
    (camino loc3 loc2)
    (camino loc2 loc4)
    (camino loc3 loc4)
    (camino loc4 loc3)
    (camino loc4 loc5)
    (camino loc5 loc4)

    ; Nuevas conexiones
    (camino loc5 loc6)    ; una sola dirección
    (camino loc6 loc7)    ; bidireccional
    (camino loc7 loc6)
  )

  (:goal
    (and
      (analizado mineral1)
      (analizado mineral2)
      (analizado mineral3)
    )
  )
)



In [6]:
with open('problem2.pddl', 'w', encoding='utf-8') as f:
    f.write(problem2_pddl.strip())
print('problem2.pddl generado correctamente.')

problem2.pddl generado correctamente.


---

## 6. Problema 3 — Escenario propuesto: zona bloqueada y dos rovers

### Escenario extendido con restricciones adicionales

- Se añade un **segundo rover** (`rover2`) que inicia en la **Localidad 4**.
- Se agrega la **Localidad 8** con `mineral_4`, conectada sólo desde `loc1` (una dirección: `loc1 → loc8`) y con regreso `loc8 → loc4`.
- La meta requiere que los cuatro minerales sean analizados, con la ventaja de la cooperación entre rovers.

```
        [loc1] → [loc8]
         ↑↓          ↓
[loc2] ← [loc3] ↔ [loc4] ↔ [loc5=LAB]
   ↓              ↑
   └──────────────┘
```

In [7]:
problem3_pddl = """
(define (problem rover-problema3)
  (:domain rover-minerales)

  (:objects
    rover1 rover2 - rover
    loc1 loc2 loc3 loc4 loc5 loc8 - localidad
    mineral1 mineral2 mineral3 mineral4 - mineral
  )

  (:init
    ; Posiciones iniciales
    (en-rover rover1 loc3)
    (manos-libres rover1)
    (en-rover rover2 loc4)
    (manos-libres rover2)

    ; Minerales
    (en-mineral mineral1 loc1)
    (en-mineral mineral2 loc2)
    (en-mineral mineral3 loc1)   ; segundo mineral en loc1
    (en-mineral mineral4 loc8)

    ; Laboratorio
    (laboratorio loc5)

    ; Red de caminos
    (camino loc3 loc1)
    (camino loc1 loc3)
    (camino loc3 loc2)
    (camino loc2 loc4)
    (camino loc3 loc4)
    (camino loc4 loc3)
    (camino loc4 loc5)
    (camino loc5 loc4)

    ; Nuevos caminos hacia loc8
    (camino loc1 loc8)     ; una sola dirección
    (camino loc8 loc4)     ; salida directa a loc4
  )

  (:goal
    (and
      (analizado mineral1)
      (analizado mineral2)
      (analizado mineral3)
      (analizado mineral4)
    )
  )
)
"""
print(problem3_pddl)


(define (problem rover-problema3)
  (:domain rover-minerales)

  (:objects
    rover1 rover2 - rover
    loc1 loc2 loc3 loc4 loc5 loc8 - localidad
    mineral1 mineral2 mineral3 mineral4 - mineral
  )

  (:init
    ; Posiciones iniciales
    (en-rover rover1 loc3)
    (manos-libres rover1)
    (en-rover rover2 loc4)
    (manos-libres rover2)

    ; Minerales
    (en-mineral mineral1 loc1)
    (en-mineral mineral2 loc2)
    (en-mineral mineral3 loc1)   ; segundo mineral en loc1
    (en-mineral mineral4 loc8)

    ; Laboratorio
    (laboratorio loc5)

    ; Red de caminos
    (camino loc3 loc1)
    (camino loc1 loc3)
    (camino loc3 loc2)
    (camino loc2 loc4)
    (camino loc3 loc4)
    (camino loc4 loc3)
    (camino loc4 loc5)
    (camino loc5 loc4)

    ; Nuevos caminos hacia loc8
    (camino loc1 loc8)     ; una sola dirección
    (camino loc8 loc4)     ; salida directa a loc4
  )

  (:goal
    (and
      (analizado mineral1)
      (analizado mineral2)
      (analizado mineral3)
  

In [8]:
with open('problem3.pddl', 'w', encoding='utf-8') as f:
    f.write(problem3_pddl.strip())
print('problem3.pddl generado correctamente.')

problem3.pddl generado correctamente.


---

## 7. Ejecución con el planificador Scorpion (IPC2018 Optimal Track Winner)

### Requisitos previos

1. **Sistema Linux** (nativo, VM o WSL en Windows con Ubuntu).
2. **Singularity** instalado (ver: https://docs.sylabs.io/guides/3.5/admin-guide/installation.html).
3. Imagen Singularity del planificador **Scorpion** descargada desde la página IPC2018.

### Comandos de ejecución

Siguiendo la sección **DETAILS ON SINGULARITY – How can I test my containers?** de la IPC2018:

```bash
# Ejecutar Problema 1
singularity run --bind $PWD:/ext scorpion.img \
    /ext/domain.pddl /ext/problem1.pddl \
    --plan-file /ext/plan1.txt

# Ejecutar Problema 2
singularity run --bind $PWD:/ext scorpion.img \
    /ext/domain.pddl /ext/problem2.pddl \
    --plan-file /ext/plan2.txt

# Ejecutar Problema 3
singularity run --bind $PWD:/ext scorpion.img \
    /ext/domain.pddl /ext/problem3.pddl \
    --plan-file /ext/plan3.txt
```

> **Nota:** Sustituir `scorpion.img` por el nombre exacto de la imagen descargada y ajustar la ruta de los archivos según el entorno.

In [9]:
# Verificar que los cuatro archivos PDDL fueron generados
import os

archivos = ['domain.pddl', 'problem1.pddl', 'problem2.pddl', 'problem3.pddl']

for archivo in archivos:
    if os.path.exists(archivo):
        size = os.path.getsize(archivo)
        print(f'  [OK] {archivo}  ({size} bytes)')
    else:
        print(f'  [FALTA] {archivo}')

  [OK] domain.pddl  (2204 bytes)
  [OK] problem1.pddl  (933 bytes)
  [OK] problem2.pddl  (989 bytes)
  [OK] problem3.pddl  (1084 bytes)


---

## 8. Análisis de los Archivos PDDL del dominio Snake (IPC2018)

Antes de codificar el problema del rover, se revisaron los archivos PDDL del dominio **Snake** disponibles en el repositorio de la IPC2018. A continuación se resume la estructura observada:

### Archivo `domain.pddl` — Snake

- Usa `:requirements :typing`.
- Tipos: `location`, `direction`.
- Predicados clave: `(is-goal ?l)`, `(connected ?l1 ?l2 ?d)`, `(occupied ?l)`, `(snake-head ?l)`, `(snake-tail ?l)`.
- Acciones: `move` (mover la cabeza del snake a la siguiente celda), considerando si la celda objetivo es meta o no.

### Archivo `problem1.pddl` — Snake

- Define una cuadrícula de localidades con conexiones en cuatro direcciones (norte, sur, este, oeste).
- El estado inicial coloca la cabeza y cola del snake en celdas específicas.
- La meta es que la cabeza alcance una celda marcada como `is-goal`.

### Diferencias con el dominio Rover

| Aspecto | Snake | Rover (esta actividad) |
|---|---|---|
| Agente | Segmento conectado (snake) | Robot unitario |
| Grafo | Cuadrícula regular | Grafo irregular con dirección |
| Objetivo | Alcanzar celda meta | Transportar y analizar minerales |
| Carga | No aplica | Minerales (uno a la vez) |

---

## 9. Conclusiones

- El lenguaje **PDDL** permite describir de forma declarativa tanto el **dominio** (tipos, predicados, acciones) como el **problema** (estado inicial y meta), separando claramente el conocimiento del dominio de la instancia concreta.
- La **asimetría de caminos** se modela de forma natural con predicados unidireccionales `(camino ?l1 ?l2)`, añadiendo o no la dirección inversa según las restricciones del terreno.
- El planificador **Scorpion** (ganador IPC2018 Optimal Track) garantiza encontrar el **plan óptimo** en número de acciones para los tres problemas propuestos.
- La extensión del dominio (más minerales, localidades, laboratorios o rovers) requiere únicamente modificar el archivo de problema, sin cambiar el dominio, lo que demuestra la **modularidad** del enfoque PDDL.

---

## Referencias

- ICAPS. (s. f.). *Competitions*. https://www.icaps-conference.org/competitions/
- IPC2018. (s. f.). *International Planning Competition 2018 – Classical Tracks*. https://ipc2018-classical.bitbucket.io/
- Sylabs. (s. f.). *Singularity Admin Guide – Installing Singularity*. https://docs.sylabs.io/guides/3.5/admin-guide/installation.html
- Seipp, J. et al. (2018). *Scorpion: A Macro-FF Planner*. IPC2018 Planner Description.